# Clustering at Scale: Overview

## Learning Objectives

1. **Define** clustering and common distance measures
2. **Contrast** hierarchical vs partitional clustering approaches
3. **Identify** why standard algorithms fail at scale
4. **Describe** evaluation metrics for clustering quality

## What is Clustering?

**Clustering:** partition $n$ points into $k$ groups (clusters) such that:
- Points in the same cluster are **similar** (close)
- Points in different clusters are **dissimilar** (far)

**Unsupervised:** no labels — we discover structure from data alone.

**Two paradigms:**
| | Hierarchical | Partitional |
|--|--|--|
| Output | Dendrogram (tree of merges) | Flat partition |
| k | Determined post-hoc | Specified upfront |
| Scalability | $O(n^2)$ to $O(n^3)$ | $O(nk)$ per iteration |
| Examples | Agglomerative, CURE | k-means, BFR |

## Distance Measures

| Distance | Formula | Use case |
|----------|---------|----------|
| Euclidean | $\sqrt{\sum_i (x_i - y_i)^2}$ | Continuous features in $\mathbb{R}^d$ |
| Manhattan | $\sum_i |x_i - y_i|$ | Grid-like spaces, robust to outliers |
| Cosine | $1 - \frac{x \cdot y}{\|x\|\|y\|}$ | High-dimensional, text/document vectors |
| Jaccard | $1 - \frac{|A \cap B|}{|A \cup B|}$ | Set-valued attributes |
| Hamming | $\sum_i \mathbb{1}[x_i \neq y_i]$ | Categorical / binary attributes |
| Edit | Min. edits to transform $x$ into $y$ | Sequences, strings |

**Choice matters:** use domain knowledge. Text → cosine; location data → Euclidean; DNA → edit distance.

In [1]:
import math
from collections import Counter

def euclidean(x, y):
    return math.sqrt(sum((a-b)**2 for a,b in zip(x,y)))

def cosine_dist(x, y):
    dot = sum(a*b for a,b in zip(x,y))
    nx = math.sqrt(sum(a**2 for a in x))
    ny = math.sqrt(sum(b**2 for b in y))
    return 1 - dot/(nx*ny) if nx*ny > 0 else 1.0

def jaccard_dist(A, B):
    a, b = set(A), set(B)
    return 1 - len(a&b)/len(a|b) if a|b else 0.0

# Example
p1 = [1,2,3,4]; p2 = [4,3,2,1]; p3 = [1,2,3,4]
print("Euclidean(p1,p2):", round(euclidean(p1,p2),3))
print("Euclidean(p1,p3):", round(euclidean(p1,p3),3))
print("Cosine dist(p1,p2):", round(cosine_dist(p1,p2),3))
print("Cosine dist(p1,p3):", round(cosine_dist(p1,p3),3))
print("Jaccard dist({1,2,3},{2,3,4}):", round(jaccard_dist([1,2,3],[2,3,4]),3))

Euclidean(p1,p2): 4.472
Euclidean(p1,p3): 0.0
Cosine dist(p1,p2): 0.333
Cosine dist(p1,p3): 0.0
Jaccard dist({1,2,3},{2,3,4}): 0.5


## Why Standard Algorithms Fail at Scale

**k-means** ($O(nkd)$ per iteration, $O(n)$ memory):
- All points must fit in memory
- Multiple passes over data

**Agglomerative** ($O(n^2)$ space, $O(n^2 \log n)$ time):
- Pairwise distance matrix does not fit for $n > 10^5$

**At scale:**
- $n = 10^9$ points, $d = 10^3$ dimensions, $k = 10^4$ clusters
- Memory: cannot hold all points at once (disk-resident)
- Passes: minimize to 1–2

**Solutions:** BFR (k-means for disk), CURE (hierarchical for disk), streaming clustering (GRGPF).

## Cluster Evaluation

### Internal Metrics (no ground truth)
| Metric | Formula | Better |
|--------|---------|--------|
| **Inertia** (WCSS) | $\sum_{c} \sum_{x \in c} d(x, \mu_c)^2$ | Lower |
| **Silhouette** | $(b-a)/\max(a,b)$ | Higher (max 1) |
| **Davies-Bouldin** | Avg $\frac{s_i + s_j}{d(\mu_i, \mu_j)}$ | Lower |

### External Metrics (with ground truth labels)
| Metric | Meaning |
|--------|---------|
| **ARI** (Adjusted Rand Index) | Corrected-for-chance cluster agreement |
| **NMI** (Normalized Mutual Info) | Information overlap between clusterings |
| **Purity** | Fraction of points that match cluster majority label |

In [2]:
def silhouette_sample(point_idx, points, labels):
    """Silhouette coefficient for a single point."""
    my_label = labels[point_idx]
    my_point = points[point_idx]
    # a: mean distance to own cluster
    same = [euclidean(my_point, points[j]) for j,l in enumerate(labels)
            if l == my_label and j != point_idx]
    if not same: return 0.0
    a = sum(same)/len(same)
    # b: min mean distance to other clusters
    other_labels = set(labels) - {my_label}
    b_vals = []
    for lbl in other_labels:
        dists = [euclidean(my_point, points[j]) for j,l in enumerate(labels) if l==lbl]
        if dists: b_vals.append(sum(dists)/len(dists))
    if not b_vals: return 0.0
    b = min(b_vals)
    return (b-a)/max(a,b)

import random
rng = random.Random(0)
# Create 3 well-separated clusters
pts, lbls = [], []
for c in range(3):
    cx, cy = (c*5, 0)
    for _ in range(20):
        pts.append([cx+rng.gauss(0,0.5), cy+rng.gauss(0,0.5)])
        lbls.append(c)

silhouettes = [silhouette_sample(i, pts, lbls) for i in range(len(pts))]
avg_s = sum(silhouettes)/len(silhouettes)
print(f"Average silhouette (3 well-separated clusters): {avg_s:.4f}")

# With bad labels (random)
bad_lbls = [rng.randint(0,2) for _ in lbls]
silhouettes_bad = [silhouette_sample(i, pts, bad_lbls) for i in range(len(pts))]
avg_bad = sum(silhouettes_bad)/len(silhouettes_bad)
print(f"Average silhouette (random labels):              {avg_bad:.4f}")


Average silhouette (3 well-separated clusters): 0.8110
Average silhouette (random labels):              -0.0792
